In [ ]:
# What is Transfer Learning?
  # Machine Learning Technique where a model trained on one task is reused (Partially/Fully) for a different but related task.
  # Transfer learning gather knowledge form a pre-trained model.

In [1]:
          # ------------How it Works----------#
# 1. Pretraining on Large Dataset
  # A model is first train on large dataset (ImageNet for Image, GPT for text).
  # The model learns general features such as edges and shapes
# 2. Fine-Tuning for a New Task
  # The pre-trained model is then adapted to anew often smaller dataset
  # Some layers may be frozen(means parapeters not updated, no learning, no training) while others are fine-tuned for the specific task.

In [ ]:
# Example of Pretrained model: VGG16, ResNet, AlexNet etc

In [3]:
# There are five steps of transfer learning
  # Import pretrained model
  # Detach the classification part of pretrained model
  # Attch the own classification layers
  # Freeze the features layers
  # Train the model

In [ ]:
# Now code Example of transfer learning using MNISt data and VGG16

In [7]:
# Some preprocessing required for mnist dataset
  # 1. reshape (28,28)
  # 2. datatype to np.uint8
  # 3. Convert 1D to 3D (3,28,28)
  # 4. Convert to PIL Image (Python Image Library)
  # 5. Resize (3,256,256)
  # 6. Center crop (3, 224, 224)
  # 7. Scaling (0,1)
  # 8. Normalize the Image


In [14]:
# Import Required Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

In [8]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [9]:
# Check GPU availability
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Available Device is: {device}')

Available Device is: cuda


In [10]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
# Extract Features values and Target values
X=df.iloc[:, 1:].to_numpy()
y=df.iloc[:, 0].to_numpy()

In [12]:
# Split the Dataset into train and testset
X_train,X_test, y_train,y_test=train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [13]:
# Perform transformations
from torchvision.transforms import transforms

custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [15]:
# Create CustomDataset Class
class CustomDataset(Dataset):

  def __init__(self, features, labels, transform):
    self.features = features
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):

    # resize to (28, 28)
    image = self.features[index].reshape(28,28)

    # change datatype to np.uint8
    image = image.astype(np.uint8)

    # change black&white to color -> (H,W,C) -> (C,H,W)
    image = np.stack([image]*3, axis=-1)

    # convert array to PIL image
    image = Image.fromarray(image)

    # apply transforms
    image = self.transform(image)

    # return
    return image, torch.tensor(self.labels[index], dtype=torch.long)

In [16]:
# Create train dataset object
train_dataset=CustomDataset(X_train, y_train, transform=custom_transform)

In [17]:
# Create test Dataset object
test_dataset=CustomDataset(X_test, y_test, transform=custom_transform)

In [18]:
# Load Dataset
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [19]:
# fetch the pretrained model

import torchvision.models as models

vgg16 = models.vgg16(pretrained=True)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 97.5MB/s]


In [20]:
# Structure of VGG16 model
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [21]:
# Freeze the features perameters
for param in vgg16.features.parameters():
  param.requires_grad=False

In [22]:
# Create own classification layers
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10)
)

In [23]:
# Transfer to cuda device
vgg16 = vgg16.to(device)

In [28]:
# Define the model
learning_rate = 0.0001
epochs = 10

In [29]:
# Loss and Optimization
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.classifier.parameters(), lr=learning_rate)

In [30]:
# training the model

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = vgg16(batch_features)

    print(outputs.shape)
    print(batch_labels.shape)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

    break

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

torch.Size([32, 10])
torch.Size([32])
Epoch: 1 , Loss: 0.001115883191426595
torch.Size([32, 10])
torch.Size([32])
Epoch: 2 , Loss: 0.0010263902346293132
torch.Size([32, 10])
torch.Size([32])
Epoch: 3 , Loss: 0.0009592890739440918
torch.Size([32, 10])
torch.Size([32])
Epoch: 4 , Loss: 0.001075997511545817
torch.Size([32, 10])
torch.Size([32])
Epoch: 5 , Loss: 0.0009269808133443196
torch.Size([32, 10])
torch.Size([32])
Epoch: 6 , Loss: 0.0008390116691589356
torch.Size([32, 10])
torch.Size([32])
Epoch: 7 , Loss: 0.0009879972139994304
torch.Size([32, 10])
torch.Size([32])
Epoch: 8 , Loss: 0.0009002350966135661
torch.Size([32, 10])
torch.Size([32])
Epoch: 9 , Loss: 0.000738047440846761
torch.Size([32, 10])
torch.Size([32])
Epoch: 10 , Loss: 0.0007449070612589519


In [31]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = vgg16(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.6506666666666666


In [32]:
# evaluation on training data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = vgg16(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.651375
